# 📘 Notebook 13: Backpropagation & Training a Network

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module D: Deep Learning · Notebook 2 of 4 in this module · (13 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Understand **loss functions** for networks and why we need them
- Explain **backpropagation** as the chain rule applied layer by layer
- Connect network training back to the **gradient descent** loop from Notebook 09
- Meet **PyTorch** and its automatic differentiation (`autograd`)
- Train your first real network in PyTorch on a small dataset

> **Prerequisites:** Notebook 12 (neurons, layers, forward pass) and Notebook 09 (gradient descent).
>
> 🔭 **The promise paid off.** In Notebook 09 we built gradient descent by hand and said *‘this same loop trains every neural network’*. This notebook makes good on that. Backpropagation is just an efficient way to compute the gradients that loop needs, across many layers.

> ℹ️ **Setup note.** This notebook introduces PyTorch. In the browser kernel, install it once:
```python
import piplite
await piplite.install(['torch', 'numpy', 'matplotlib'])
```
PyTorch is a large package; the install may take a little while in the browser. If `torch` is unavailable in your environment, the NumPy sections still run, and the PyTorch code can be run later in a standard Python setup.

## 1. Measuring how wrong the network is: the loss function

### The idea
Before a network can improve, it needs a number that says how bad its current predictions are. That number is the **loss** (or cost). You already met one: the Mean Squared Error from Notebook 09. The choice of loss depends on the task:

- **Regression** → **Mean Squared Error**: $\;L = \frac{1}{n}\sum (\hat{y}_i - y_i)^2$
- **Classification** → **Cross-Entropy**, which measures the distance between predicted probabilities and the true labels.

Training has one goal: **adjust the weights to make the loss as small as possible**, minimising a cost function, exactly as in Notebook 09, just with far more parameters.

In [ ]:
import numpy as np

def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)

y_true = np.array([1.0, 0.0, 1.0])
good = np.array([0.9, 0.1, 0.8])
bad  = np.array([0.2, 0.7, 0.3])
print("Loss (good predictions):", round(mse_loss(good, y_true), 3))
print("Loss (bad predictions): ", round(mse_loss(bad,  y_true), 3))

## 2. Backpropagation: the chain rule, organised

### The problem it solves
Gradient descent needs the gradient of the loss with respect to **every** weight, including weights buried deep inside the network. How does changing a weight in the first layer affect the final loss, when its influence passes through every later layer? The answer is the **chain rule** from calculus, applied step by step backwards through the network.

### Intuition first
After a forward pass produces a prediction and a loss, **backpropagation** works *backwards* from the loss to the inputs, computing at each layer how much that layer contributed to the error. It is called *back*propagation because the error signal propagates from the output back toward the input. Each weight then learns its share of the blame, its gradient.

### The training loop (identical in spirit to Notebook 09)
1. **Forward pass:** compute predictions.
2. **Compute loss:** how wrong were we?
3. **Backward pass (backprop):** compute the gradient of the loss for every weight.
4. **Update:** nudge every weight downhill: $\; w \leftarrow w - \alpha \frac{\partial L}{\partial w}$.
5. **Repeat** for many epochs.

Steps 1, 4, and 5 are precisely Notebook 09's loop. Backprop is just how step 3 is done efficiently for deep models.

> 🧠 **Why this was a breakthrough.** Computing these gradients naively would be hopelessly slow for large networks. Backpropagation reuses intermediate results so the whole gradient is obtained in essentially one backward sweep. This efficiency is what made training deep networks practical at all.

## 3. PyTorch and automatic differentiation

### Why a framework
Deriving and coding gradients by hand for every architecture would be error-prone and tedious. **PyTorch** does it automatically: you write only the **forward** computation, and its **autograd** engine works out all the gradients for you. This is the single biggest reason deep-learning frameworks exist.

### Tensors
PyTorch's core object is the **tensor**, essentially a NumPy array that can also live on a GPU and can track gradients. If you understood NumPy arrays (Notebook 05), you already understand tensors.

In [ ]:
import torch

# A tensor that tracks gradients
x = torch.tensor(3.0, requires_grad=True)

# Define a simple function y = x^2 + 2x
y = x**2 + 2*x

# autograd computes dy/dx for us
y.backward()
print(f"y = {y.item()}")
print(f"dy/dx at x=3 is {x.grad.item()}  (analytically 2x+2 = 8)")

PyTorch computed the derivative automatically, no calculus by hand. For a network with millions of weights, it does exactly this for every one of them. That is autograd doing backpropagation on your behalf.

## 4. Training your first PyTorch network

We will rebuild the small network from Notebook 12: but in PyTorch, and this time it will actually **learn**. The task: learn the XOR-like pattern, a classic problem that a linear model *cannot* solve but a network *can* (thanks to the non-linear activations from Notebook 12).

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)   # reproducibility

# XOR data: output is 1 when inputs differ
X = torch.tensor([[0.,0.], [0.,1.], [1.,0.], [1.,1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

# Define the network as a class inheriting from nn.Module
# (exactly the structure previewed in Notebook 04 and built by hand in Notebook 12)
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)   # 2 inputs -> 4 hidden neurons
        self.output = nn.Linear(4, 1)   # 4 hidden -> 1 output

    def forward(self, x):
        x = torch.relu(self.hidden(x))  # hidden layer + ReLU
        x = torch.sigmoid(self.output(x))  # output as a probability
        return x

model = XORNet()
print(model)

> 🔭 **Everything here is familiar.** `class XORNet(nn.Module)` is the inheritance from Notebook 04. `nn.Linear(2, 4)` is the $W\mathbf{x}+\mathbf{b}$ layer from Notebook 05/12. `torch.relu` and `torch.sigmoid` are the activations from Notebook 12. `forward` is the forward pass you wrote by hand. PyTorch only adds the automatic gradients.

In [ ]:
# The training ingredients
loss_fn = nn.BCELoss()                                  # binary cross-entropy
optimizer = torch.optim.SGD(model.parameters(), lr=0.5) # gradient descent!

losses = []
for epoch in range(2000):
    optimizer.zero_grad()      # reset gradients
    y_pred = model(X)          # 1. forward pass
    loss = loss_fn(y_pred, y)  # 2. compute loss
    loss.backward()            # 3. backprop: autograd computes all gradients
    optimizer.step()           # 4. update weights (the downhill step)
    losses.append(loss.item())

print(f"Final loss: {losses[-1]:.4f}")
print("Predictions after training:")
print(model(X).detach().round())

The four lines inside the loop, `zero_grad`, `backward`, `step`, plus the forward pass, are the universal PyTorch training loop. Compare them to Notebook 09's hand-written loop: same five steps, now with autograd doing the gradient maths. Let us watch the loss fall:

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(losses)
plt.title("Training loss decreasing over epochs")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.show()

> ⚠️ **Common pitfalls**
>
> - Forgetting `optimizer.zero_grad()`, PyTorch *accumulates* gradients by default, so without resetting they pile up and training breaks.
> - Forgetting `.detach()` (or `torch.no_grad()`) when inspecting predictions, otherwise you keep tracking gradients unnecessarily.
> - A learning rate that is too high or too low fails here for the same reasons as in Notebook 09. The principles carry over unchanged.

---
## ✏️ Exercises

### Exercise 1
Use PyTorch autograd to compute the derivative of $y = x^3$ at $x = 2$. Confirm it matches the analytical answer $3x^2 = 12$.

In [ ]:
import torch
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
x = torch.tensor(2.0, requires_grad=True)
y = x**3
y.backward()
print(x.grad.item())   # 12.0
```
</details>

### Exercise 2
In the XOR training loop, change the optimiser learning rate from `0.5` to `0.01` and keep 2000 epochs. Does the network still learn XOR within that budget? Relate your observation to the learning-rate discussion in Notebook 09.

In [ ]:
# Modify and re-run the training loop with lr=0.01, then inspect the final loss.


<details><summary>💡 Show solution</summary>

```python
# With lr=0.01 the loss decreases far more slowly and often does NOT reach a good
# solution within 2000 epochs -- the steps downhill are too small. This is exactly
# the 'learning rate too low -> slow convergence' behaviour observed in Notebook 09.
```

*The intuition is identical to classical gradient descent because it *is* gradient descent.*
</details>

## 🔑 Key takeaways

- A **loss function** quantifies how wrong the network is (MSE for regression, cross-entropy for classification).
- **Backpropagation** applies the chain rule backwards through the network to get every weight's gradient efficiently.
- Training is the **same gradient-descent loop** from Notebook 09: forward → loss → backward → update → repeat.
- **PyTorch tensors** are NumPy-like arrays with autograd; **autograd** does backprop automatically.
- The PyTorch training loop (`zero_grad`, `backward`, `step`) is the practical realisation of everything in Modules C-D.

---
**Next:** *Notebook 14: Convolutional Neural Networks*, the architecture that revolutionised computer vision.

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*